# P-ML9 — Strategy Integration: `RegimeLGBMStrategy`

**Goal:** Wrap the P-ML7 `RegimeEnsemble` (16-feature, Sharpe +1.261) into the `BaseStrategy`
interface so the ML signal can be backtested with the standard engine, combined with risk
overlays, and eventually deployed.

**Two objectives:**
1. **Validation:** `RegimeLGBMStrategy` in binary mode must reproduce the P-ML7 walk-forward
   equity curve exactly (same predictions ↔ same signals ↔ same equity).
2. **Scaled positioning:** Test `scale_positions=True` which converts raw predictions to a
   rolling z-score before clipping — reduces position size on low-confidence predictions
   and may improve Sharpe or reduce MaxDD.

**Strategy design:**
- `RegimeLGBMStrategy(ensemble, regime_classifier, feature_columns, ...)` wraps a
  **pre-trained** `RegimeEnsemble` (training is still done per-fold in the walk-forward loop).
- `generate_signals(df)` builds features internally from raw OHLCV, then predicts.
- `predict(X, regime)` is a fast path accepting pre-computed features (used in §3 validation).
- Scaled mode: `signal = clip(pred_zscore × 0.5, −1, +1)` where `pred_zscore` is a
  60-bar rolling z-score of raw predictions.

## §1 — Config

In [ ]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         9,
})

# -- Dataset / walk-forward config (identical to P-ML7) ------------------------
SYMBOL     = "BTC/USDT"
SINCE      = "2019-01-01"
UNTIL      = "2025-01-01"
HORIZON    = 1
N_SPLITS   = 5
TRAIN_FRAC = 0.6
PURGE      = 1
LONG_MA    = 200
ADX_THRESH = 25.0
MIN_BULL_BARS = 30

# -- Feature set (P-ML7 FEATURES_V2, 16 features) -----------------------------
FEATURES_V2 = [
    "bar_ret", "bb_zscore", "rsi", "macd_hist_norm", "atr_pct",
    "bb_width", "upper_wick", "lower_wick", "hl_range",
    "vol_log_chg", "di_diff", "adx",
    "ret_5", "ret_20", "mom_zscore_20", "ret_5_minus_20",
]

# -- Scaled positioning parameters ---------------------------------------------
PRED_ZSCORE_WINDOW = 60    # rolling window for prediction z-score
POSITION_SCALE     = 0.5   # multiplier: pred_zscore * 0.5 before clip
MAX_POSITION       = 1.0   # clip to [-1, +1]

# -- P-ML7 reference metrics (from F14) ----------------------------------------
P_ML7_ICS    = [0.0721, 0.0091, 0.1283, 0.0537, 0.1069]
BUY_HOLD     = {"return": 2.996, "sharpe": 1.379, "maxdd": -0.354}
P_ML7        = {"return": 19.976, "sharpe": 1.261, "maxdd": -0.773}

print(f"Dataset:          {SINCE} -> {UNTIL} | 1d | horizon={HORIZON}")
print(f"Walk-forward:     {N_SPLITS} folds, train_frac={TRAIN_FRAC}, purge={PURGE}")
print(f"Feature set:      FEATURES_V2 ({len(FEATURES_V2)} features)")
print(f"Scaled mode:      pred_zscore_window={PRED_ZSCORE_WINDOW}, scale={POSITION_SCALE}, max={MAX_POSITION}")
print(f"\nP-ML7 reference:  Sharpe={P_ML7['sharpe']:+.3f}, Return={P_ML7['return']*100:+.1f}%, MaxDD={P_ML7['maxdd']*100:.1f}%")

## §2 — Data Loading & Feature Construction

In [ ]:
from data.fetch import fetch_ohlcv
from ml.features import build_feature_matrix
from ml.features.momentum import build_momentum_features
from ml.labels import forward_return
from ml.regime import RegimeClassifier
from ml.validation import purged_wf_splits
from ml.models import RegimeEnsemble
from strategies.ml import RegimeLGBMStrategy
from backtesting import compute_metrics

# Cache-guard (same as P-ML7)
df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d", since=SINCE, until=UNTIL)
if df_raw.index.min() > pd.Timestamp("2020-01-01", tz="UTC"):
    print("Cache missing early data -- re-fetching...")
    df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d",
                         since=SINCE, until=UNTIL, use_cache=False)
print(f"Loaded {len(df_raw):,} bars  |  {df_raw.index[0].date()} -> {df_raw.index[-1].date()}")

# Build features + labels + regimes (identical to P-ML7 setup)
feats_base = build_feature_matrix(df_raw)
feats_mom  = build_momentum_features(df_raw)
label      = forward_return(df_raw, horizon=HORIZON)
rc         = RegimeClassifier(long_ma=LONG_MA, adx_thresh=ADX_THRESH)
reg        = rc.transform(df_raw)

comb = pd.concat([feats_base, feats_mom, label, reg["regime"]], axis=1).dropna()

X_v2  = comb[FEATURES_V2]
y_all = comb[label.name]
regime_all = comb["regime"].fillna("ranging")

bar_ret_daily = np.log(df_raw["close"] / df_raw["close"].shift(1)).reindex(comb.index)
splits = list(purged_wf_splits(len(comb), N_SPLITS, TRAIN_FRAC, purge_bars=PURGE))

print(f"\n{len(comb):,} usable bars | {comb.index[0].date()} -> {comb.index[-1].date()}")
print(f"Splits: {N_SPLITS} folds")
print(f"\nRegimeLGBMStrategy imported: {RegimeLGBMStrategy}")

## §3 — Walk-Forward Validation: Binary Mode

Run the same 5-fold walk-forward as P-ML7, but via `RegimeLGBMStrategy`:
1. Train `RegimeEnsemble` per fold (identical to P-ML7)
2. Wrap in `RegimeLGBMStrategy` with `scale_positions=False` (binary signals)
3. Call `strat.predict(X_v2.iloc[te], regime_all.iloc[te])` — pre-computed feature fast-path
4. Verify: OOS IC and equity must match P-ML7 within floating-point tolerance

In [ ]:
fold_results_binary = []

header = (f"{'Fold':<5} {'Period':<35} {'n_train':>8} {'bull_tr':>7} "
          f"{'P7 IC (ref)':>12} {'P9 IC':>8} {'Match?':>8}")
print(header)
print("-" * 90)

for i, (tr, te) in enumerate(splits):
    # Train ensemble (identical to P-ML7)
    ensemble = RegimeEnsemble(min_bull_bars=MIN_BULL_BARS)
    ensemble.fit(X_v2.iloc[tr], y_all.iloc[tr], regime_all.iloc[tr])

    # Wrap in strategy (binary mode)
    strat = RegimeLGBMStrategy(
        ensemble=ensemble,
        regime_classifier=rc,
        feature_columns=FEATURES_V2,
        scale_positions=False,
    )

    # Predict via strategy.predict (pre-computed feature fast-path)
    preds  = strat.predict(X_v2.iloc[te], regime_all.iloc[te])
    actual = y_all.iloc[te].values

    rho, _ = stats.spearmanr(preds, actual)
    p7_ic  = P_ML7_ICS[i]

    # Signal from strategy helper
    signals = strat.signal_from_predictions(preds, X_v2.iloc[te].index)

    n_bull_tr = int((regime_all.iloc[tr] == "bull").sum())
    period    = f"{comb.index[te[0]].date()} -> {comb.index[te[-1]].date()}"
    match_str = "OK" if abs(rho - p7_ic) < 1e-6 else f"DELTA={rho - p7_ic:+.2e}"

    print(f"  {i+1:<3} {period:<35} {len(tr):>8} {n_bull_tr:>7} "
          f"{p7_ic:>+12.4f} {rho:>+8.4f} {match_str:>8}")

    fold_results_binary.append({
        "fold": i + 1, "te": te, "IC": rho, "preds": preds,
        "signals": signals, "ensemble": ensemble, "strat": strat,
    })

ics_binary = [r["IC"] for r in fold_results_binary]
print()
print(f"P-ML9 (binary): Mean IC={np.mean(ics_binary):+.4f}  "
      f"ICIR={np.mean(ics_binary)/np.std(ics_binary):.3f}")
print(f"P-ML7 (ref):    Mean IC={np.mean(P_ML7_ICS):+.4f}  "
      f"ICIR={np.mean(P_ML7_ICS)/np.std(P_ML7_ICS):.3f}")
print()
ic_match = all(abs(r["IC"] - P_ML7_ICS[i]) < 1e-6 for i, r in enumerate(fold_results_binary))
if ic_match:
    print("ALL ICs MATCH P-ML7 EXACTLY -- strategy correctly encapsulates P-ML7 logic")
else:
    print("ICs DO NOT MATCH -- investigate discrepancy")

In [ ]:
# -- Equity curve validation: P-ML9 (binary) vs P-ML7 -------------------------
def build_equity(fold_preds_list, bar_ret_daily, idx):
    pieces, anchor = [], 1.0
    for te, preds in fold_preds_list:
        pos   = np.sign(preds)
        ret   = bar_ret_daily.iloc[te].values
        eq    = np.cumprod(1 + np.roll(pos, 1) * ret)
        eq[0] = 1.0
        s = pd.Series(eq, index=idx[te])
        s = s / s.iloc[0] * anchor
        anchor = float(s.iloc[-1])
        pieces.append(s)
    return pd.concat(pieces)


idx = comb.index

# Binary equity curve (should match P-ML7)
oos_binary = build_equity(
    [(r["te"], r["preds"]) for r in fold_results_binary],
    bar_ret_daily, idx,
)
bah = df_raw["close"].reindex(oos_binary.index)
bah = bah / bah.iloc[0]

m_binary = compute_metrics(oos_binary)

print(f"\n{'Metric':<30} {'P-ML7 (F14 ref)':>16} {'P-ML9 binary':>14}")
print("-" * 62)
print(f"  {'OOS Sharpe':<28} {P_ML7['sharpe']:>+16.3f} {m_binary['sharpe_ratio']:>+14.3f}")
print(f"  {'OOS Return':<28} {P_ML7['return']*100:>+15.1f}% {m_binary['total_return']*100:>+13.1f}%")
print(f"  {'Max Drawdown':<28} {P_ML7['maxdd']*100:>+15.1f}% {m_binary['max_drawdown']*100:>+13.1f}%")
print()
sharpe_match = abs(m_binary['sharpe_ratio'] - P_ML7['sharpe']) < 0.01
if sharpe_match:
    print("Equity curve matches P-ML7 (delta Sharpe < 0.01)")
else:
    print(f"Delta Sharpe = {m_binary['sharpe_ratio']-P_ML7['sharpe']:+.4f}")

## §4 — Walk-Forward: Scaled Positioning

Re-run the same walk-forward with `scale_positions=True`:
- `signal = clip(pred_zscore × 0.5, −1, +1)`
- `pred_zscore` = rolling 60-bar z-score of raw ensemble predictions

Hypothesis: scaled positioning reduces whipsaw losses on low-confidence predictions,
improving risk-adjusted return (Sharpe) and potentially reducing MaxDD.

In [ ]:
fold_results_scaled = []

header_sc = (f"{'Fold':<5} {'Period':<35} {'Binary IC':>10} {'Scaled IC':>10} "
             f"{'delta IC':>7} {'hit_bin':>8} {'hit_sc':>8}")
print(header_sc)
print("-" * 90)

for i, (tr, te) in enumerate(splits):
    # Reuse already-trained ensemble from binary walk-forward
    ensemble = fold_results_binary[i]["ensemble"]

    # Wrap in strategy (scaled mode)
    strat_sc = RegimeLGBMStrategy(
        ensemble=ensemble,
        regime_classifier=rc,
        feature_columns=FEATURES_V2,
        scale_positions=True,
        pred_zscore_window=PRED_ZSCORE_WINDOW,
        position_scale=POSITION_SCALE,
        max_position=MAX_POSITION,
    )

    preds  = strat_sc.predict(X_v2.iloc[te], regime_all.iloc[te])
    actual = y_all.iloc[te].values

    rho_sc, _ = stats.spearmanr(preds, actual)   # same IC as binary (same preds)
    signals_sc = strat_sc.signal_from_predictions(preds, X_v2.iloc[te].index)

    # Hit rate (fraction where sign(scaled signal) matches sign(actual))
    hit_bin = (np.sign(fold_results_binary[i]["preds"]) == np.sign(actual)).mean()
    hit_sc  = (np.sign(signals_sc) == np.sign(actual)).mean()

    ic_bin = fold_results_binary[i]["IC"]
    period = f"{comb.index[te[0]].date()} -> {comb.index[te[-1]].date()}"

    print(f"  {i+1:<3} {period:<35} {ic_bin:>+10.4f} {rho_sc:>+10.4f} "
          f"{rho_sc - ic_bin:>+7.4f} {hit_bin:>8.3f} {hit_sc:>8.3f}")

    fold_results_scaled.append({
        "fold": i + 1, "te": te, "IC": rho_sc,
        "preds": preds, "signals_sc": signals_sc, "strat": strat_sc,
    })

print()
print("Note: IC is identical between binary and scaled modes because")
print("      scaled mode uses the same raw predictions -- only position sizing differs.")

In [ ]:
# -- Build scaled equity curve -------------------------------------------------
def build_equity_scaled(fold_results, bar_ret_daily, idx):
    """Build equity curve from scaled (continuous) position signals."""
    pieces, anchor = [], 1.0
    for r in fold_results:
        te       = r["te"]
        pos      = r["signals_sc"].values       # continuous in [-max, +max]
        ret      = bar_ret_daily.iloc[te].values
        # Shift position by 1 bar to avoid look-ahead (bar open-to-close)
        pos_sh   = np.roll(pos, 1)
        pos_sh[0] = 0
        eq       = np.cumprod(1 + pos_sh * ret)
        eq[0]    = 1.0
        s = pd.Series(eq, index=idx[te])
        s = s / s.iloc[0] * anchor
        anchor = float(s.iloc[-1])
        pieces.append(s)
    return pd.concat(pieces)


oos_scaled = build_equity_scaled(fold_results_scaled, bar_ret_daily, idx)
m_scaled   = compute_metrics(oos_scaled)

# -- Equity comparison plot ----------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))
oos_binary.plot(ax=ax, label="P-ML9 binary (+1/-1/0)",
                color="#9b59b6", linewidth=2.0, alpha=0.85)
oos_scaled.plot(ax=ax, label=f"P-ML9 scaled (zscore x {POSITION_SCALE}, clip +/-{MAX_POSITION})",
                color="#e67e22", linewidth=2.0, alpha=0.85)
bah.plot(ax=ax, label="Buy-and-Hold",
         color="black", linewidth=1.0, linestyle=":")
for _, (tr, te) in enumerate(splits):
    ax.axvline(idx[te[0]], color="lightgray", linewidth=0.5, linestyle="--")
ax.axhline(1, color="black", linewidth=0.4)
ax.set_title("OOS Equity -- P-ML9: Binary vs Scaled Positioning", fontsize=11)
ax.set_ylabel("Equity (normalised)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# -- Drawdown comparison -------------------------------------------------------
def rolling_drawdown(eq):
    roll_max = eq.cummax()
    return (eq - roll_max) / roll_max

fig, ax = plt.subplots(figsize=(14, 3))
rolling_drawdown(oos_binary).plot(ax=ax, label="Binary",
                                   color="#9b59b6", linewidth=1.5, alpha=0.85)
rolling_drawdown(oos_scaled).plot(ax=ax, label="Scaled",
                                   color="#e67e22", linewidth=1.5, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.4)
ax.set_title("Underwater Equity (Drawdown) -- Binary vs Scaled", fontsize=10)
ax.set_ylabel("Drawdown")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# -- Metrics table -------------------------------------------------------------
print(f"\n{'Strategy':<40} {'Return':>9} {'Sharpe':>8} {'MaxDD':>8}")
print("-" * 68)
rows = [
    ("Buy & Hold",                        BUY_HOLD["return"],       BUY_HOLD["sharpe"],   BUY_HOLD["maxdd"]),
    ("P-ML7 reference (from F14)",        P_ML7["return"],         P_ML7["sharpe"],      P_ML7["maxdd"]),
    ("P-ML9 binary (strategy class)",     m_binary["total_return"], m_binary["sharpe_ratio"], m_binary["max_drawdown"]),
    (f"P-ML9 scaled (x{POSITION_SCALE}, clip+/-{MAX_POSITION})",
                                          m_scaled["total_return"], m_scaled["sharpe_ratio"], m_scaled["max_drawdown"]),
]
for name, ret, shr, mdd in rows:
    print(f"  {name:<38} {ret*100:>+8.1f}%  {shr:>+7.3f}  {mdd*100:>7.1f}%")

## §5 — Full-Pipeline Test: `generate_signals(df_ohlcv)`

Validate the **OHLCV mode** of `generate_signals(df)` — where the strategy builds features
internally from raw OHLCV bars. This tests the full production pipeline.

Method: for the final fold, train ensemble on training OHLCV → `generate_signals(test_ohlcv)`
→ compare predictions to pre-computed fast-path predictions from §3.

In [ ]:
# Use fold 5 (latest, most realistic OOS window)
fold_idx = 4   # 0-indexed
tr, te   = splits[fold_idx]

# -- OHLCV indices for train and test ------------------------------------------
train_ts = comb.index[tr]
test_ts  = comb.index[te]

# For generate_signals we need raw OHLCV.
# To compute features for test bars, we prepend warmup bars from df_raw.
# Need >= 250 bars: SMA(200) for RegimeClassifier + ADX(14) + momentum(60+20).
warmup_n   = 250
test_start = test_ts[0]
test_end   = test_ts[-1]

# Find position of test_start in df_raw and prepend warmup bars
df_raw_idx    = df_raw.index.get_loc(test_start)
warmup_start  = max(0, df_raw_idx - warmup_n)
df_test_ohlcv = df_raw.iloc[warmup_start : df_raw.index.get_loc(test_end) + 1]

print(f"Fold {fold_idx+1} | Train: {train_ts[0].date()} -> {train_ts[-1].date()} ({len(tr)} bars)")
print(f"         Test:  {test_ts[0].date()} -> {test_ts[-1].date()} ({len(te)} bars)")
print(f"         Test OHLCV (with {warmup_n} warmup): {len(df_test_ohlcv)} bars")

# -- Reuse already-trained fold-5 ensemble -------------------------------------
ensemble_f5 = fold_results_binary[fold_idx]["ensemble"]

# -- Generate signals from raw OHLCV (full pipeline) ---------------------------
strat_ohlcv = RegimeLGBMStrategy(
    ensemble=ensemble_f5,
    regime_classifier=rc,
    feature_columns=FEATURES_V2,
    scale_positions=False,
)

sig_df = strat_ohlcv.generate_signals(df_test_ohlcv)

# -- Restrict to the actual OOS timestamps -------------------------------------
sig_oos = sig_df.loc[test_ts]
preds_ohlcv_path = sig_oos["pred"].values

# -- Compare to pre-computed predictions from S3 --------------------------------
preds_fastpath = fold_results_binary[fold_idx]["preds"]

max_diff = np.abs(preds_ohlcv_path - preds_fastpath).max()
signal_match = (sig_oos["signal"].values == np.sign(preds_fastpath)).all()

print(f"\nMax |pred difference| (OHLCV path vs fast-path): {max_diff:.2e}")
print(f"Signal column matches binary fast-path: {signal_match}")

# Sample
print(f"\nFirst 5 OOS bars (signal from raw OHLCV path):")
print(sig_oos[["close", "pred", "regime", "signal"]].head().to_string())

## §6 — Finding F16 Summary

In [ ]:
bin_return = m_binary["total_return"]
bin_sharpe = m_binary["sharpe_ratio"]
bin_maxdd  = m_binary["max_drawdown"]
sc_return  = m_scaled["total_return"]
sc_sharpe  = m_scaled["sharpe_ratio"]
sc_maxdd   = m_scaled["max_drawdown"]

binary_matches_p7   = abs(bin_sharpe  - P_ML7["sharpe"]) < 0.01
ohlcv_path_works    = signal_match
scaled_reduces_dd   = sc_maxdd   > bin_maxdd     # less negative = better
scaled_better_sharpe= sc_sharpe  > bin_sharpe

print("=" * 72)
print("FINDING F16 -- Strategy Integration (P-ML9)")
print("=" * 72)
print()
print("RegimeLGBMStrategy class created in strategies/ml/regime_lgbm.py")
print(f"  - feature_columns: {len(FEATURES_V2)} (FEATURES_V2 from P-ML7)")
print(f"  - scale_positions: False (binary) or True (pred_zscore x {POSITION_SCALE})")
print()
print(f"{'Metric':<40} {'P-ML7 (F14)':>12} {'Binary mode':>12} {'Scaled mode':>12}")
print("-" * 78)
print(f"  {'OOS Sharpe':<38} {P_ML7['sharpe']:>+12.3f} {bin_sharpe:>+12.3f} {sc_sharpe:>+12.3f}")
print(f"  {'OOS Return':<38} {P_ML7['return']*100:>+11.1f}% {bin_return*100:>+11.1f}% {sc_return*100:>+11.1f}%")
print(f"  {'Max Drawdown':<38} {P_ML7['maxdd']*100:>+11.1f}% {bin_maxdd*100:>+11.1f}% {sc_maxdd*100:>+11.1f}%")
print()
if binary_matches_p7:
    print("Binary mode reproduces P-ML7?    YES")
else:
    print("Binary mode reproduces P-ML7?    NO -- investigate")
if ohlcv_path_works:
    print("OHLCV pipeline works?             YES")
else:
    print("OHLCV pipeline works?             NO -- investigate")
if scaled_reduces_dd:
    print("Scaled mode reduces MaxDD?        YES")
else:
    print("Scaled mode reduces MaxDD?        NO")
if scaled_better_sharpe:
    print("Scaled mode improves Sharpe?      YES")
else:
    print("Scaled mode improves Sharpe?      NO")
print()

if scaled_better_sharpe and scaled_reduces_dd:
    scaled_verdict = (
        f"SCALED MODE IMPROVES ON BINARY: Sharpe {sc_sharpe:+.3f} vs {bin_sharpe:+.3f}, "
        f"MaxDD {sc_maxdd*100:.1f}% vs {bin_maxdd*100:.1f}%. "
        "Confidence-weighted sizing reduces whipsaw losses without discarding the IC signal."
    )
elif scaled_reduces_dd:
    scaled_verdict = (
        f"SCALED MODE REDUCES MAXDD ({sc_maxdd*100:.1f}% vs {bin_maxdd*100:.1f}%) "
        f"BUT HURTS SHARPE ({sc_sharpe:+.3f} vs {bin_sharpe:+.3f}). "
        "Scaled positions reduce both upside and downside. A higher position_scale "
        "(e.g. 0.8) or different window might recover more return."
    )
elif scaled_better_sharpe:
    scaled_verdict = (
        f"SCALED MODE IMPROVES SHARPE ({sc_sharpe:+.3f} vs {bin_sharpe:+.3f}) "
        f"BUT WORSENS MAXDD ({sc_maxdd*100:.1f}% vs {bin_maxdd*100:.1f}%). "
        "The volatility-timing benefit of smaller positions on uncertain predictions "
        "is offset by compounding losses during large drawdown periods."
    )
else:
    scaled_verdict = (
        f"SCALED MODE UNDERPERFORMS BINARY on both Sharpe and MaxDD. "
        "The pred_zscore rolling window may be too long for the daily signal autocorrelation, "
        "or POSITION_SCALE=0.5 reduces positions too aggressively. "
        "P-ML10 (dedicated risk overlay) should explore drawdown-gated sizing instead."
    )

print(f"Scaled mode verdict: {scaled_verdict}")
print()
print("Next steps:")
print("  - P-ML10: Risk overlay -- drawdown brake + bull cap (separate from position scaling)")
print("    to address MaxDD -77.3% without hurting return as much as simple scaling")